<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/treaper33-hue/ML-AI-assisted-/blob/main/compilers_and_triton/notebooks/a3_passes.ipynb)

# A3 — Folding, propagation, CSE and DCE, to a fixpoint

**INTERMEDIATE** · runtime a second or two

---

Four classical passes — constant folding, copy propagation, common
subexpression elimination and dead-code elimination — run to a fixpoint,
because each one creates work for the others. Correctness is established by
**differential testing**: the optimised program is run against the
original on hundreds of random inputs.

---

## How to use this notebook

1. Run **Setup**. It defines the stage and runs nothing.
2. Run the stage and read its output, gates included.
3. Work through the sections below, which take the pieces apart and
   invite you to break them.

The gates are declared *before* the run. When one fails the useful question
is which is wrong — the code, or the threshold someone wrote down before the
problem was understood.

## Setup

Defines everything. Runs nothing.

In [ ]:
#@title Setup — run this cell first (collapsed: it is the whole stage)
# ===== embedded: track_a_compilers/a1_lexer_parser.py =====
"""A1 · FUNDAMENTAL — From text to a tree.

The first thing any compiler does, and the only part most people ever
write by hand. A lexer turns characters into tokens; a parser turns
tokens into a tree that encodes precedence and associativity so that
nothing downstream has to think about either again.

The language is deliberately tiny — arithmetic, comparisons, variables,
`let` bindings and calls — because the *shape* of the pipeline is what
transfers, not the grammar. Every later stage in this track consumes the
tree this module produces.

Why a Pratt parser: precedence climbing expresses operator precedence as
data (a table of binding powers) rather than as a tower of mutually
recursive functions. Adding an operator is a one-line change. This is
the same technique production parsers use.

Run it:
    python a1_lexer_parser.py
"""

import dataclasses
import re
import sys
from typing import Dict, List, Sequence, Tuple, Union

# --------------------------------------------------------------------- #
#                              1. Tokens                                 #
# --------------------------------------------------------------------- #

TOKEN_SPEC: List[Tuple[str, str]] = [
    ('NUMBER', r'\d+\.\d+|\d+'),
    ('NAME', r'[A-Za-z_][A-Za-z_0-9]*'),
    ('OP', r'<=|>=|==|!=|[-+*/%<>()=,]'),
    ('SKIP', r'[ \t]+'),
    ('MISMATCH', r'.'),
]
TOKEN_RE = re.compile('|'.join(f'(?P<{name}>{pattern})'
                               for name, pattern in TOKEN_SPEC))
KEYWORDS = {'let', 'in'}


@dataclasses.dataclass(frozen=True)
class Token:
    """One lexical token.

    Attributes:
        kind: 'NUMBER', 'NAME', 'KEYWORD', 'OP' or 'EOF'.
        text: The matched source text.
        position: Index into the source, for error messages.
    """

    kind: str
    text: str
    position: int

    def __repr__(self) -> str:
        """Returns a compact form for test output."""
        return f'{self.kind}({self.text!r})'


class CompileError(Exception):
    """Raised for any error in the front end, with a source position."""

    def __init__(self, message: str, position: int, source: str = '') -> None:
        """Builds an error that can point at the offending character.

        Args:
            message: What went wrong.
            position: Index into the source.
            source: The source text, for the caret line.
        """
        self.position = position
        if source:
            caret = ' ' * position + '^'
            message = f'{message}\n  {source}\n  {caret}'
        super().__init__(message)


def tokenize(source: str) -> List[Token]:
    """Converts source text into a token list.

    Args:
        source: The program text.

    Returns:
        Tokens, ending with a single EOF token.

    Raises:
        CompileError: On any character the language does not define.
    """
    tokens: List[Token] = []
    for match in TOKEN_RE.finditer(source):
        kind = match.lastgroup
        text = match.group()
        if kind == 'SKIP':
            continue
        if kind == 'MISMATCH':
            raise CompileError(f'unexpected character {text!r}',
                               match.start(), source)
        if kind == 'NAME' and text in KEYWORDS:
            kind = 'KEYWORD'
        tokens.append(Token(kind, text, match.start()))
    tokens.append(Token('EOF', '', len(source)))
    return tokens


# --------------------------------------------------------------------- #
#                            2. The tree                                 #
# --------------------------------------------------------------------- #


@dataclasses.dataclass(frozen=True)
class Number:
    """A literal."""

    value: float


@dataclasses.dataclass(frozen=True)
class Name:
    """A variable reference."""

    identifier: str


@dataclasses.dataclass(frozen=True)
class Unary:
    """Prefix negation."""

    operator: str
    operand: 'Node'


@dataclasses.dataclass(frozen=True)
class Binary:
    """An infix operation."""

    operator: str
    left: 'Node'
    right: 'Node'


@dataclasses.dataclass(frozen=True)
class Call:
    """A function call."""

    callee: str
    arguments: Tuple['Node', ...]


@dataclasses.dataclass(frozen=True)
class Let:
    """A binding: `let name = value in body`."""

    name: str
    value: 'Node'
    body: 'Node'


Node = Union[Number, Name, Unary, Binary, Call, Let]

# Binding power drives precedence. Higher binds tighter. Expressing it
# as data is the whole point: adding an operator is one line here, not a
# new level of recursive descent.
BINDING_POWER: Dict[str, int] = {
    '==': 10, '!=': 10, '<': 10, '>': 10, '<=': 10, '>=': 10,
    '+': 20, '-': 20,
    '*': 30, '/': 30, '%': 30,
}
RIGHT_ASSOCIATIVE: set = set()


class Parser:
    """A Pratt (precedence-climbing) parser.

    Attributes:
        tokens: The token stream.
        source: Original text, for error messages.
        index: Current position in the stream.
    """

    def __init__(self, tokens: Sequence[Token], source: str = '') -> None:
        """Stores the stream.

        Args:
            tokens: Tokens from `tokenize`.
            source: The original text.
        """
        self.tokens = list(tokens)
        self.source = source
        self.index = 0

    @property
    def current(self) -> Token:
        """The token under the cursor."""
        return self.tokens[self.index]

    def advance(self) -> Token:
        """Consumes and returns the current token."""
        token = self.current
        self.index += 1
        return token

    def expect(self, text: str) -> Token:
        """Consumes a token that must have the given text.

        Args:
            text: The expected token text.

        Returns:
            The consumed token.

        Raises:
            CompileError: If the next token does not match.
        """
        if self.current.text != text:
            raise CompileError(
                f'expected {text!r}, found {self.current.text!r}',
                self.current.position, self.source)
        return self.advance()

    def parse(self) -> Node:
        """Parses one complete expression.

        Returns:
            The root of the tree.

        Raises:
            CompileError: If tokens remain after the expression.
        """
        node = self.expression(0)
        if self.current.kind != 'EOF':
            raise CompileError(f'unexpected {self.current.text!r}',
                               self.current.position, self.source)
        return node

    def expression(self, minimum_power: int) -> Node:
        """Parses an expression, stopping below a binding power.

        This single loop is the whole precedence mechanism: keep
        absorbing operators while they bind at least as tightly as the
        caller allows, and recurse with a higher floor for the
        right-hand side.

        Args:
            minimum_power: The caller's binding power floor.

        Returns:
            The parsed subtree.
        """
        left = self.prefix()
        while True:
            token = self.current
            power = BINDING_POWER.get(token.text, 0)
            if token.kind != 'OP' or power < minimum_power or power == 0:
                break
            self.advance()
            next_floor = (power if token.text in RIGHT_ASSOCIATIVE
                          else power + 1)
            right = self.expression(next_floor)
            left = Binary(token.text, left, right)
        return left

    def prefix(self) -> Node:
        """Parses a prefix position: literal, name, call, group or `let`.

        Returns:
            The parsed subtree.

        Raises:
            CompileError: On anything that cannot begin an expression.
        """
        token = self.advance()

        if token.kind == 'NUMBER':
            return Number(float(token.text))

        if token.kind == 'KEYWORD' and token.text == 'let':
            name = self.advance()
            if name.kind != 'NAME':
                raise CompileError('expected a name after `let`',
                                   name.position, self.source)
            self.expect('=')
            value = self.expression(0)
            keyword = self.advance()
            if keyword.text != 'in':
                raise CompileError('expected `in` after the bound value',
                                   keyword.position, self.source)
            return Let(name.text, value, self.expression(0))

        if token.kind == 'NAME':
            if self.current.text == '(':
                self.advance()
                arguments: List[Node] = []
                if self.current.text != ')':
                    arguments.append(self.expression(0))
                    while self.current.text == ',':
                        self.advance()
                        arguments.append(self.expression(0))
                self.expect(')')
                return Call(token.text, tuple(arguments))
            return Name(token.text)

        if token.text == '(':
            inner = self.expression(0)
            self.expect(')')
            return inner

        if token.text == '-':
            return Unary('-', self.expression(max(BINDING_POWER.values()) + 1))

        raise CompileError(f'{token.text!r} cannot start an expression',
                           token.position, self.source)


def parse(source: str) -> Node:
    """Parses source text into a tree.

    Args:
        source: The program.

    Returns:
        The root node.
    """
    return Parser(tokenize(source), source).parse()


def render(node: Node) -> str:
    """Renders a tree back to fully parenthesised source.

    Round-tripping through this is how the gates below check that
    precedence was parsed the way the language claims.

    Args:
        node: Any node.

    Returns:
        A parenthesised string.
    """
    if isinstance(node, Number):
        value = node.value
        return str(int(value)) if value == int(value) else str(value)
    if isinstance(node, Name):
        return node.identifier
    if isinstance(node, Unary):
        return f'(-{render(node.operand)})'
    if isinstance(node, Binary):
        return f'({render(node.left)} {node.operator} {render(node.right)})'
    if isinstance(node, Call):
        inner = ', '.join(render(a) for a in node.arguments)
        return f'{node.callee}({inner})'
    if isinstance(node, Let):
        return (f'(let {node.name} = {render(node.value)} '
                f'in {render(node.body)})')
    raise TypeError(f'unknown node {type(node).__name__}')


GATES: List[Tuple[str, str, str]] = [
    ('precedence', '1 + 2 * 3', '(1 + (2 * 3))'),
    ('left associativity', '1 - 2 - 3', '((1 - 2) - 3)'),
    ('parentheses override', '(1 + 2) * 3', '((1 + 2) * 3)'),
    ('unary binds tightest', '-2 * 3', '((-2) * 3)'),
    ('comparison is loosest', 'a + 1 < b * 2',
     '((a + 1) < (b * 2))'),
    ('calls', 'max(a, b + 1)', 'max(a, (b + 1))'),
    ('let', 'let x = 2 in x * x', '(let x = 2 in (x * x))'),
    ('nested let', 'let x = 1 in let y = 2 in x + y',
     '(let x = 1 in (let y = 2 in (x + y)))'),
]


def main() -> int:
    """Demonstrates the front end and checks the gates.

    Returns:
        0 when every gate passed.
    """
    print('=' * 70)
    print('A1 · FUNDAMENTAL — From text to a tree')
    print('=' * 70)

    sample = 'let scale = 2 in max(scale * x + 1, y - 3)'
    print(f'\nsource : {sample}')
    print('tokens :', ' '.join(repr(t) for t in tokenize(sample)[:8]), '...')
    print(f'tree   : {render(parse(sample))}')

    print('\nGates — precedence and associativity, checked by round-trip')
    failures = 0
    for name, source, expected in GATES:
        actual = render(parse(source))
        ok = actual == expected
        failures += not ok
        print(f'  [{"PASS" if ok else "FAIL"}] {name:<24} '
              f'{source:<34} -> {actual}')
        if not ok:
            print(f'{"":9}expected {expected}')

    print('\nErrors point at the character')
    for bad in ('1 + * 2', 'let = 3 in 1', '2 $ 3'):
        try:
            parse(bad)
            print(f'  [FAIL] {bad!r} should not have parsed')
            failures += 1
        except CompileError as error:
            first = str(error).splitlines()[0]
            print(f'  [PASS] {bad!r:<16} {first}')

    print(f'\nstatus: {"VERIFIED" if not failures else "FAILED"}')
    return 0 if not failures else 1


# ===== embedded: track_a_compilers/a2_ir.py =====
"""A2 · FUNDAMENTAL — From a tree to an instruction list.

A tree is good for parsing and bad for optimising. Optimisation wants a
*flat* sequence where every value has a name and every operation has
exactly one result, because then "is this value used again?" and "have I
computed this before?" are questions you can answer by looking, rather
than by walking a tree.

That representation is **three-address code**: `t3 = mul t1 t2`. Each
instruction names its destination and its operands, and every temporary
is assigned exactly once — static single assignment, the property that
makes the passes in A3 straightforward instead of subtle.

Two things to notice:

* `let` disappears. Scoping was a source-language concern; after
  lowering there are only names, and the environment that mapped them
  is gone. Lowering is where language features turn into machine
  concepts.
* The interpreter here is a *specification*. Every optimisation in A3
  must preserve what it computes, and the gates check exactly that.

Run it:
    python a2_ir.py
"""

import dataclasses
import math
import sys
from typing import Callable, Dict, List, Optional, Tuple, Union



BUILTINS: Dict[str, Tuple[int, Callable]] = {
    'max': (2, max),
    'min': (2, min),
    'exp': (1, math.exp),
    'log': (1, math.log),
    'sqrt': (1, math.sqrt),
    'abs': (1, abs),
}

BINARY_OPS: Dict[str, Callable] = {
    '+': lambda a, b: a + b,
    '-': lambda a, b: a - b,
    '*': lambda a, b: a * b,
    '/': lambda a, b: a / b,
    '%': lambda a, b: a % b,
    '<': lambda a, b: float(a < b),
    '>': lambda a, b: float(a > b),
    '<=': lambda a, b: float(a <= b),
    '>=': lambda a, b: float(a >= b),
    '==': lambda a, b: float(a == b),
    '!=': lambda a, b: float(a != b),
}

# Operations with no side effects, whose results may be folded, shared
# or deleted when unused. Division is included but guarded: folding a
# division by zero at compile time would turn a runtime error into a
# compile-time crash, which is a change in behaviour, not an
# optimisation.
PURE = set(BINARY_OPS) | set(BUILTINS) | {'neg', 'const', 'copy'}


@dataclasses.dataclass(frozen=True)
class Instruction:
    """One three-address instruction.

    Attributes:
        op: 'const', 'copy', 'neg', a binary operator, or a builtin name.
        dest: The temporary this writes, assigned exactly once.
        args: Operand names, or a single literal for 'const'.
    """

    op: str
    dest: str
    args: Tuple[Union[str, float], ...]

    def __str__(self) -> str:
        """Renders the instruction in assembly-like form."""
        if self.op == 'const':
            return f'{self.dest} = {self.args[0]}'
        if self.op == 'copy':
            return f'{self.dest} = {self.args[0]}'
        if self.op == 'neg':
            return f'{self.dest} = neg {self.args[0]}'
        if self.op in BINARY_OPS:
            return f'{self.dest} = {self.args[0]} {self.op} {self.args[1]}'
        return f'{self.dest} = {self.op}({", ".join(map(str, self.args))})'


@dataclasses.dataclass
class Program:
    """A lowered program.

    Attributes:
        instructions: The instruction list, in order.
        result: The temporary holding the final value.
        free_names: Variables the program reads but does not define.
    """

    instructions: List[Instruction]
    result: str
    free_names: Tuple[str, ...] = ()

    def __str__(self) -> str:
        """Renders the whole program."""
        body = '\n'.join(f'  {instruction}'
                         for instruction in self.instructions)
        return f'{body}\n  return {self.result}'

    def count(self) -> int:
        """Returns the instruction count, the headline size metric."""
        return len(self.instructions)


class Lowerer:
    """Walks a tree and emits three-address code.

    Attributes:
        instructions: Emitted so far.
        environment: Source names mapped to temporaries.
        free_names: Names read without being bound.
    """

    def __init__(self) -> None:
        """Starts an empty program."""
        self.instructions: List[Instruction] = []
        self.environment: Dict[str, str] = {}
        self.free_names: List[str] = []
        self._counter = 0

    def fresh(self) -> str:
        """Returns a new, never-reused temporary name."""
        name = f't{self._counter}'
        self._counter += 1
        return name

    def emit(self, op: str, *args: Union[str, float]) -> str:
        """Appends one instruction and returns its destination.

        Args:
            op: The operation.
            *args: Operands.

        Returns:
            The destination temporary.
        """
        dest = self.fresh()
        self.instructions.append(Instruction(op, dest, tuple(args)))
        return dest

    def lower(self, node: Node) -> str:
        """Lowers a subtree, returning the temporary holding its value.

        Args:
            node: The subtree.

        Returns:
            The destination temporary.

        Raises:
            CompileError: On an unknown builtin or wrong arity.
        """
        if isinstance(node, Number):
            return self.emit('const', node.value)

        if isinstance(node, Name):
            if node.identifier in self.environment:
                return self.environment[node.identifier]
            # A free variable: an input to the program.
            if node.identifier not in self.free_names:
                self.free_names.append(node.identifier)
            return node.identifier

        if isinstance(node, Unary):
            return self.emit('neg', self.lower(node.operand))

        if isinstance(node, Binary):
            left = self.lower(node.left)
            right = self.lower(node.right)
            return self.emit(node.operator, left, right)

        if isinstance(node, Call):
            if node.callee not in BUILTINS:
                raise CompileError(f'unknown function {node.callee!r}', 0)
            arity, _ = BUILTINS[node.callee]
            if len(node.arguments) != arity:
                raise CompileError(
                    f'{node.callee} takes {arity} argument(s), '
                    f'got {len(node.arguments)}', 0)
            lowered = [self.lower(a) for a in node.arguments]
            return self.emit(node.callee, *lowered)

        if isinstance(node, Let):
            value = self.lower(node.value)
            # Bind, lower the body, then restore -- lexical scoping,
            # enforced here and nowhere else downstream.
            shadowed = self.environment.get(node.name)
            self.environment[node.name] = value
            result = self.lower(node.body)
            if shadowed is None:
                self.environment.pop(node.name, None)
            else:
                self.environment[node.name] = shadowed
            return result

        raise TypeError(f'cannot lower {type(node).__name__}')


def lower(node: Node) -> Program:
    """Lowers a tree into a program.

    Args:
        node: The root.

    Returns:
        The lowered program.
    """
    lowerer = Lowerer()
    result = lowerer.lower(node)
    return Program(lowerer.instructions, result,
                   tuple(lowerer.free_names))


def compile_source(source: str) -> Program:
    """Parses and lowers in one step.

    Args:
        source: The program text.

    Returns:
        The lowered program.
    """
    return lower(parse(source))


def execute(program: Program,
            inputs: Optional[Dict[str, float]] = None) -> float:
    """Interprets a program.

    This is the specification every optimisation must preserve: A3
    checks each pass by running the program before and after and
    comparing results on random inputs.

    Args:
        program: The program to run.
        inputs: Values for its free variables.

    Returns:
        The value of the result temporary.

    Raises:
        CompileError: If a free variable has no value.
    """
    environment: Dict[str, float] = dict(inputs or {})
    for name in program.free_names:
        if name not in environment:
            raise CompileError(f'no value supplied for {name!r}', 0)

    def read(operand: Union[str, float]) -> float:
        """Resolves an operand to a number."""
        if isinstance(operand, (int, float)):
            return float(operand)
        return environment[operand]

    for instruction in program.instructions:
        if instruction.op == 'const':
            environment[instruction.dest] = float(instruction.args[0])
        elif instruction.op == 'copy':
            environment[instruction.dest] = read(instruction.args[0])
        elif instruction.op == 'neg':
            environment[instruction.dest] = -read(instruction.args[0])
        elif instruction.op in BINARY_OPS:
            environment[instruction.dest] = BINARY_OPS[instruction.op](
                read(instruction.args[0]), read(instruction.args[1]))
        elif instruction.op in BUILTINS:
            _, function = BUILTINS[instruction.op]
            environment[instruction.dest] = float(
                function(*[read(a) for a in instruction.args]))
        else:
            raise CompileError(f'unknown op {instruction.op!r}', 0)
    return float(environment[program.result])


def evaluate_tree(node: Node,
                  inputs: Optional[Dict[str, float]] = None) -> float:
    """Evaluates a tree directly, without lowering.

    An independent second implementation. Agreement between this and
    `execute` is what makes the lowering gate meaningful: two routes to
    the same number, one of them not involving the IR at all.

    Args:
        node: The tree.
        inputs: Values for free variables.

    Returns:
        The value.
    """
    environment = dict(inputs or {})

    def walk(current: Node) -> float:
        """Recursively evaluates one node."""
        if isinstance(current, Number):
            return current.value
        if isinstance(current, Name):
            return environment[current.identifier]
        if isinstance(current, Unary):
            return -walk(current.operand)
        if isinstance(current, Binary):
            return BINARY_OPS[current.operator](walk(current.left),
                                                walk(current.right))
        if isinstance(current, Call):
            _, function = BUILTINS[current.callee]
            return float(function(*[walk(a) for a in current.arguments]))
        if isinstance(current, Let):
            previous = environment.get(current.name)
            environment[current.name] = walk(current.value)
            result = walk(current.body)
            if previous is None:
                environment.pop(current.name, None)
            else:
                environment[current.name] = previous
            return result
        raise TypeError(f'cannot evaluate {type(current).__name__}')

    return walk(node)


SAMPLES: List[Tuple[str, Dict[str, float]]] = [
    ('1 + 2 * 3', {}),
    ('let x = 4 in x * x + x', {}),
    ('max(a * 2, b + 1)', {'a': 3.0, 'b': 10.0}),
    ('let s = a + b in s * s - s', {'a': 2.0, 'b': 5.0}),
    ('exp(0) + sqrt(16) - abs(-3)', {}),
    ('let x = 2 in let x = x + 1 in x * 10', {}),
]


def main() -> int:
    """Demonstrates lowering and checks the gates.

    Returns:
        0 when every gate passed.
    """
    print('=' * 70)
    print('A2 · FUNDAMENTAL — From a tree to an instruction list')
    print('=' * 70)

    source = 'let s = a + b in s * s - s'
    program = compile_source(source)
    print(f'\nsource:\n  {source}')
    print(f'\nthree-address code (free: {", ".join(program.free_names)}):')
    print(program)
    print('\nNote that `let` has vanished: s was bound to t0, and every')
    print('later use is just t2. Scoping was a front-end concern.')

    print('\nGates — the IR computes what the tree computes')
    failures = 0
    for text, inputs in SAMPLES:
        tree = parse(text)
        direct = evaluate_tree(tree, inputs)
        lowered = compile_source(text)
        through_ir = execute(lowered, inputs)
        ok = abs(direct - through_ir) < 1e-12
        failures += not ok
        print(f'  [{"PASS" if ok else "FAIL"}] {text:<38} '
              f'tree={direct:<10.4f} ir={through_ir:<10.4f} '
              f'({lowered.count()} instructions)')

    print('\nGate — every temporary is assigned exactly once (SSA)')
    for text, _ in SAMPLES:
        lowered = compile_source(text)
        destinations = [i.dest for i in lowered.instructions]
        ok = len(destinations) == len(set(destinations))
        failures += not ok
        if not ok:
            print(f'  [FAIL] {text} reassigns a temporary')
    print(f'  [{"PASS" if not failures else "FAIL"}] '
          f'checked {len(SAMPLES)} programs')

    print('\nGate — shadowing resolves to the inner binding')
    shadow = execute(compile_source('let x = 2 in let x = x + 1 in x * 10'))
    ok = abs(shadow - 30.0) < 1e-12
    failures += not ok
    print(f'  [{"PASS" if ok else "FAIL"}] inner x wins: {shadow} (expect 30)')

    print(f'\nstatus: {"VERIFIED" if not failures else "FAILED"}')
    return 0 if not failures else 1


# ===== track_a_compilers/a3_passes.py =====
"""A3 · INTERMEDIATE — Optimisation passes that preserve meaning.

Four classical passes, each a few dozen lines, run to a fixpoint:

* **Constant folding** — evaluate what is already known.
* **Copy propagation** — remove the indirection `b = a` introduces.
* **Common subexpression elimination** — compute each value once.
* **Dead code elimination** — delete what nothing reads.

They are individually simple and collectively powerful, because each
one creates opportunities for the others: folding makes copies, copies
expose common subexpressions, eliminating them makes instructions dead.
That mutual feeding is why the driver loops until nothing changes
rather than running each pass once.

**The discipline that matters more than the passes.** An optimisation
is only correct if the program still computes the same thing, and the
only way to believe that is to check it. Every gate here runs the
program before and after on random inputs and compares. That is
differential testing, and it is how real compilers are tested, because
proving a pass correct is hard and running it on a thousand inputs is
easy.

Run it:
    python a3_passes.py
"""

import math
import random
import sys
from typing import Callable, Dict, List, Set, Tuple, Union




def _is_literal(value: Union[str, float]) -> bool:
    """Whether an operand is an immediate number."""
    return isinstance(value, (int, float))


def constant_folding(program: Program) -> Tuple[Program, int]:
    """Evaluates instructions whose operands are all known constants.

    Division is folded only when the divisor is non-zero: folding a
    division by zero would turn a runtime error into a compile-time
    crash, which changes behaviour rather than preserving it. Compilers
    that get this wrong break programs that never execute the faulty
    branch.

    Args:
        program: The program to fold.

    Returns:
        A tuple (new program, number of instructions folded).
    """
    known: Dict[str, float] = {}
    output: List[Instruction] = []
    folded = 0

    for instruction in program.instructions:
        operands = [known.get(a, a) if isinstance(a, str) else a
                    for a in instruction.args]

        if instruction.op == 'const':
            known[instruction.dest] = float(instruction.args[0])
            output.append(instruction)
            continue

        all_known = all(_is_literal(o) for o in operands)
        if all_known and instruction.op in PURE:
            try:
                if instruction.op in BINARY_OPS:
                    if (instruction.op in ('/', '%')
                            and float(operands[1]) == 0.0):
                        raise ZeroDivisionError
                    value = BINARY_OPS[instruction.op](float(operands[0]),
                                                       float(operands[1]))
                elif instruction.op == 'neg':
                    value = -float(operands[0])
                elif instruction.op == 'copy':
                    value = float(operands[0])
                elif instruction.op in BUILTINS:
                    _, function = BUILTINS[instruction.op]
                    value = float(function(*[float(o) for o in operands]))
                else:
                    raise ValueError
            except (ZeroDivisionError, ValueError, OverflowError):
                # Not foldable at compile time: leave it for runtime,
                # where the error belongs.
                output.append(instruction)
                continue
            known[instruction.dest] = value
            output.append(Instruction('const', instruction.dest, (value,)))
            folded += 1
        else:
            output.append(instruction)

    return Program(output, program.result, program.free_names), folded


def copy_propagation(program: Program) -> Tuple[Program, int]:
    """Replaces reads of a copy with reads of the original.

    Args:
        program: The program.

    Returns:
        A tuple (new program, number of operands rewritten).
    """
    alias: Dict[str, str] = {}
    output: List[Instruction] = []
    rewritten = 0

    for instruction in program.instructions:
        args = []
        for operand in instruction.args:
            if isinstance(operand, str) and operand in alias:
                args.append(alias[operand])
                rewritten += 1
            else:
                args.append(operand)
        args_tuple = tuple(args)

        if instruction.op == 'copy' and isinstance(args_tuple[0], str):
            alias[instruction.dest] = args_tuple[0]
        output.append(Instruction(instruction.op, instruction.dest,
                                  args_tuple))

    result = alias.get(program.result, program.result)
    return Program(output, result, program.free_names), rewritten


def common_subexpression_elimination(
        program: Program) -> Tuple[Program, int]:
    """Replaces a recomputation with a copy of the earlier result.

    Only pure operations are shared. The key is the operation together
    with its operand names, which is sound here precisely because the
    IR is single-assignment: a name's value never changes, so two
    instructions with the same key must produce the same value.

    Commutative operations use a sorted key, so `a + b` and `b + a`
    hash together.

    Args:
        program: The program.

    Returns:
        A tuple (new program, number of recomputations removed).
    """
    seen: Dict[Tuple, str] = {}
    output: List[Instruction] = []
    removed = 0
    commutative = {'+', '*', 'max', 'min', '==', '!='}

    for instruction in program.instructions:
        if instruction.op not in PURE or instruction.op == 'const':
            output.append(instruction)
            continue
        operands = instruction.args
        if instruction.op in commutative:
            operands = tuple(sorted(operands, key=repr))
        key = (instruction.op, operands)
        if key in seen:
            output.append(Instruction('copy', instruction.dest,
                                      (seen[key],)))
            removed += 1
        else:
            seen[key] = instruction.dest
            output.append(instruction)

    return Program(output, program.result, program.free_names), removed


def dead_code_elimination(program: Program) -> Tuple[Program, int]:
    """Removes instructions whose results nothing reads.

    Liveness is computed backwards from the result: a value is live if
    the result depends on it, directly or transitively. Walking
    backwards means one pass suffices for a straight-line program.

    Args:
        program: The program.

    Returns:
        A tuple (new program, number of instructions deleted).
    """
    live: Set[str] = {program.result}
    keep: List[Instruction] = []

    for instruction in reversed(program.instructions):
        if instruction.dest not in live:
            if instruction.op in PURE:
                continue                    # pure and unused: delete it
        keep.append(instruction)
        live.discard(instruction.dest)
        for operand in instruction.args:
            if isinstance(operand, str):
                live.add(operand)

    keep.reverse()
    return (Program(keep, program.result, program.free_names),
            program.count() - len(keep))


PASSES: List[Tuple[str, Callable[[Program], Tuple[Program, int]]]] = [
    ('constant folding', constant_folding),
    ('copy propagation', copy_propagation),
    ('common subexpression elimination',
     common_subexpression_elimination),
    ('dead code elimination', dead_code_elimination),
]


def optimise(program: Program, max_rounds: int = 10,
             trace: bool = False) -> Tuple[Program, List[str]]:
    """Runs every pass to a fixpoint.

    Each pass creates work for the others, so one round is never
    enough. The loop stops when a full round changes nothing, which is
    the definition of a fixpoint and also the only honest termination
    condition.

    Args:
        program: The program to optimise.
        max_rounds: Safety bound; reaching it means a pass is unstable.
        trace: Whether to record a line per effective pass.

    Returns:
        A tuple (optimised program, trace lines).
    """
    log: List[str] = []
    for round_number in range(1, max_rounds + 1):
        changed = 0
        for name, pass_function in PASSES:
            before = program.count()
            program, count = pass_function(program)
            changed += count
            if trace and count:
                log.append(f'round {round_number}: {name:<34} '
                           f'{count:>2} change(s), '
                           f'{before} -> {program.count()} instructions')
        if changed == 0:
            if trace:
                log.append(f'round {round_number}: fixpoint reached')
            break
    return program, log


BENCHMARKS: List[Tuple[str, str]] = [
    ('redundant work', 'let s = a + b in s * s + (a + b) * (a + b)'),
    ('constants', '2 * 3 + 4 * 5 - 6 / 2'),
    ('mixed', 'let k = 2 * 8 in max(a * k, a * k) + exp(0)'),
    ('deep reuse', '(a + b) * (a + b) + (a + b) * (a + b)'),
    ('dead values', 'let unused = a * b * 999 in a + 1'),
    ('builtin folding', 'sqrt(16) + abs(-4) + min(3, 9)'),
]


def differential_check(before: Program, after: Program, trials: int = 200,
                       seed: int = 20260101) -> Tuple[bool, str]:
    """Checks two programs agree on random inputs.

    Args:
        before: The original program.
        after: The optimised program.
        trials: How many random input vectors to try.
        seed: Seed for reproducibility.

    Returns:
        A tuple (agreed, detail).
    """
    rng = random.Random(seed)
    names = sorted(set(before.free_names) | set(after.free_names))
    worst = 0.0
    for _ in range(trials):
        inputs = {name: rng.uniform(-50.0, 50.0) for name in names}
        try:
            expected = execute(before, inputs)
            actual = execute(after, inputs)
        except (ZeroDivisionError, ValueError, OverflowError):
            continue                        # same input, same refusal
        if math.isnan(expected) and math.isnan(actual):
            continue
        scale = max(1.0, abs(expected))
        worst = max(worst, abs(expected - actual) / scale)
    return worst < 1e-12, f'worst relative difference {worst:.2e}'


def main() -> int:
    """Demonstrates the passes and checks the gates.

    Returns:
        0 when every gate passed.
    """
    print('=' * 70)
    print('A3 · INTERMEDIATE — Optimisation passes that preserve meaning')
    print('=' * 70)

    source = 'let s = a + b in s * s + (a + b) * (a + b)'
    original = compile_source(source)
    print(f'\nsource:\n  {source}')
    print(f'\nbefore ({original.count()} instructions):')
    print(original)

    optimised, log = optimise(original, trace=True)
    print('\npass log:')
    for line in log:
        print(f'  {line}')
    print(f'\nafter ({optimised.count()} instructions):')
    print(optimised)
    print('\nThe three copies of (a + b) became one. CSE could prove they')
    print('were the same value because single assignment guarantees a')
    print('name never changes meaning.')

    failures = 0
    print('\nGates — size reduced AND meaning preserved')
    print(f'  {"benchmark":<20}{"before":>8}{"after":>8}{"saved":>8}  check')
    for name, text in BENCHMARKS:
        start = compile_source(text)
        end, _ = optimise(start)
        agreed, detail = differential_check(start, end)
        smaller = end.count() <= start.count()
        ok = agreed and smaller
        failures += not ok
        saved = start.count() - end.count()
        print(f'  {name:<20}{start.count():>8}{end.count():>8}'
              f'{saved:>8}  [{"PASS" if ok else "FAIL"}] {detail}')

    print('\nGate — the pipeline is idempotent')
    for name, text in BENCHMARKS:
        once, _ = optimise(compile_source(text))
        twice, _ = optimise(once)
        if once.count() != twice.count():
            print(f'  [FAIL] {name}: {once.count()} -> {twice.count()}')
            failures += 1
    print(f'  [{"PASS" if not failures else "FAIL"}] '
          f'a second run changes nothing')

    print('\nGate — division by zero is NOT folded')
    risky = compile_source('1 / 0')
    folded, _ = constant_folding(risky)
    still_divides = any(i.op == '/' for i in folded.instructions)
    failures += not still_divides
    print(f'  [{"PASS" if still_divides else "FAIL"}] '
          f'the division survives to runtime, where the error belongs')

    print('\nGate — a free variable is never folded away')
    kept = optimise(compile_source('a * 1 + 0'))[0]
    reads_a = any('a' in [str(x) for x in i.args]
                  for i in kept.instructions)
    failures += not reads_a
    print(f'  [{"PASS" if reads_a else "FAIL"}] '
          f'`a` is still read (we do not implement algebraic identities)')

    print(f'\nstatus: {"VERIFIED" if not failures else "FAILED"}')
    return 0 if not failures else 1

## Run it

The stage from start to finish, with its gates at the end.

In [ ]:
main()

## One program, pass by pass

The trace shows why a single round is never enough: folding creates
copies, propagation makes subexpressions identical, CSE makes
instructions dead, DCE removes them — and that can expose more folding.
The loop stops when a whole round changes nothing, which is the only
honest termination condition.

In [ ]:
program = compile_source(
    'let s = a + b in s * s + (a + b) * (a + b)')
print('BEFORE')
print(program)
optimised, trace = optimise(program, trace=True)
print('\nTRACE')
for line in trace:
    print(' ', line)
print('\nAFTER')
print(optimised)

## The fixpoint is not decoration

Cap the rounds at one and work is left on the table. The middle column
is what a pipeline that runs each pass once achieves; the right-hand one
is convergence.

In [ ]:
for label, source in BENCHMARKS:
    base = compile_source(source)
    one, _ = optimise(compile_source(source), max_rounds=1)
    full, _ = optimise(compile_source(source))
    print(f'{label:<22} {base.count():>3} -> '
          f'{one.count():>3} (one round) -> '
          f'{full.count():>3} (fixpoint)')

## Differential testing, which is how this is really checked

Proving a pass correct is hard; running the optimised program against the
original on a thousand random inputs is easy, and it is what real
compiler test suites do. Raise the trial count and it stays silent —
that silence is the evidence.

In [ ]:
for label, source in BENCHMARKS:
    before = compile_source(source)
    after, _ = optimise(compile_source(source))
    ok, detail = differential_check(before, after,
                                    trials=1000)
    verdict = 'agree' if ok else 'DISAGREE'
    print(f'{label:<22} {verdict:<9} {detail}')

## The optimisation that must not happen

Folding `1 / 0` would turn a runtime error into a compile-time crash.
That is a change in behaviour, not an optimisation — the compiler would be
wrong even though the program it produced was smaller.

In [ ]:
program = compile_source('1 / 0')
folded, _ = optimise(program)
print('IR after optimisation:')
print(folded)
print('\nthe division survived, so the error happens at '
      'run time where it belongs:')
try:
    execute(folded, {})
except ZeroDivisionError as error:
    print(f'  ZeroDivisionError: {error}')

---

## Where this sits

The script version is `track_a_compilers/a3_passes.py`, and
`compilers_and_triton/run_all.py` runs the whole ladder in order.
`compilers_and_triton/README.md` explains why the compiler track and the
Triton track are one course rather than two.